# Kernel Derivation for the Exponential Envelope (Eq. 10)

This notebook derives $\hat{\Gamma}(\omega)$ and $R_\Gamma(\tau)$ for the **piecewise exponential**
spot emergence/decay profile (Eq. 10, `alpha_exp`):

$$\alpha_{\rm exp}(t) = \alpha_{\max} \times
\begin{cases}
    e^{t/\tau_{\rm em}} & t < 0 \quad\text{(exponential rise)} \\
    1 & 0 \leq t \leq \tau_{\rm plat} \quad\text{(plateau)} \\
    e^{-(t - \tau_{\rm plat})/\tau_{\rm dec}} & t > \tau_{\rm plat} \quad\text{(exponential decay)}
\end{cases}$$

where we set $t_1 = 0$, $t_2 = \tau_{\rm plat}$ without loss of generality.

**Steps:**
1. Derive $\hat{\Gamma}(\omega)$ analytically via direct integration of $\Gamma(t) = \alpha^2(t)$ (this notebook keeps the factor $\alpha_{\max}^2$; the paper normalizes $\Gamma = \alpha^2/\alpha_{\max}^2$)
2. Derive $R_\Gamma(\tau)$ analytically via direct convolution
3. Verify both against numerical computation
4. Assemble the full single-spot kernel and compare to the trapezoidal kernel

**Key result:** Both $\hat{\Gamma}(\omega)$ and $R_\Gamma(\tau)$ have closed-form expressions,
but $R_\Gamma(\tau)$ has kinks (discontinuous first derivative) at $\tau = 0$ and $\tau = \tau_{\rm plat}$
due to the $C^0$ nature of the envelope.

In [ ]:
import sympy as sp
from sympy import pi, cos, sin, exp, integrate, simplify, symbols, latex, oo, Rational, Abs
from IPython.display import display, Math
sp.init_printing()

In [ ]:
t = sp.Symbol('t', real=True)
s = sp.Symbol('s', real=True)
w = sp.Symbol('omega', positive=True)
T = sp.Symbol('T', positive=True)  # lag variable (tau)
tau_e = sp.Symbol('tau_e', positive=True)  # emergence timescale
tau_d = sp.Symbol('tau_d', positive=True)  # decay timescale
tau_p = sp.Symbol('tau_plat', positive=True)  # plateau duration
amax = sp.Symbol('alpha_max', positive=True)

## 1. The Squared Envelope $\Gamma(t) = \alpha_{\rm exp}^2(t)$

Squaring the piecewise exponential preserves the piecewise structure with doubled rates:

$$\Gamma(t) = \alpha_{\max}^2 \times
\begin{cases}
    e^{2t/\tau_{\rm em}} & t < 0 \\
    1 & 0 \leq t \leq \tau_{\rm plat} \\
    e^{-2(t - \tau_{\rm plat})/\tau_{\rm dec}} & t > \tau_{\rm plat}
\end{cases}$$

This is $C^0$ but not $C^1$: the first derivative is discontinuous at $t = 0$ and $t = \tau_{\rm plat}$.

### Check: $\alpha_{\rm exp}(t)$ and $\Gamma(t)$ as typeset in the paper

Take `eq:alpha_exp` (Section `sec:spot_profile`, here with $t_1 = 0$, $t_2 = \tau_{\rm plat}$) and the
normalized squared envelope `eq:Gamma_exp` of Section `subsec:ft_envelope`, item (2), from the equation
registry (`equations.py`), where they are transcribed exactly as written in `ms.tex`. The paper normalizes
$\Gamma = \alpha^2/\alpha_{\max}^2$, while this notebook keeps $\Gamma = \alpha^2$. Assert that the paper's
$\Gamma(t) = \alpha_{\rm exp}^2(t)/\alpha_{\max}^2$,
and that $\Gamma$ is $C^0$ but not $C^1$ at $t = 0$ and $t = \tau_{\rm plat}$, as stated above.

In [ ]:
# eq:alpha_exp and eq:Gamma_exp as typeset, from the equation registry (equations.py)
import equations as eqs

alpha_exp_paper = eqs.get("eq:alpha_exp").expr.subs({eqs.t1: 0, eqs.t2: tau_p})  # t_1 = 0, t_2 = tau_plat
Gamma_paper = eqs.get("eq:Gamma_exp").expr      # normalized: alpha_exp^2 / alpha_max^2

assert simplify(sp.piecewise_fold(Gamma_paper - alpha_exp_paper**2 / amax**2)) == 0, \
    "paper's Gamma(t) is not alpha_exp(t)^2 / alpha_max^2"

# Branches of the paper's Gamma(t): rise (t<0), plateau, decay (t>tau_plat)
_Gamma_paper_pw = sp.piecewise_fold(Gamma_paper)
assert isinstance(_Gamma_paper_pw, sp.Piecewise) and len(_Gamma_paper_pw.args) == 3
G_rise_paper, G_plat_paper, G_dec_paper = [e for e, _ in _Gamma_paper_pw.args]

# C^0: branches agree at t = 0 and t = tau_plat
assert simplify(G_rise_paper.subs(t, 0) - G_plat_paper) == 0
assert simplify(G_dec_paper.subs(t, tau_p) - G_plat_paper) == 0
# not C^1: one-sided slopes jump by -2/tau_em at t=0 and -2/tau_dec at t=tau_plat
slope_jump_0 = simplify(sp.diff(G_plat_paper, t) - sp.diff(G_rise_paper, t).subs(t, 0))
slope_jump_tm = simplify(sp.diff(G_dec_paper, t).subs(t, tau_p) - sp.diff(G_plat_paper, t))
assert simplify(slope_jump_0 + 2 / tau_e) == 0
assert simplify(slope_jump_tm + 2 / tau_d) == 0
print("Verified: Gamma(t) = alpha_exp(t)^2 / alpha_max^2 (paper); C^0 with slope jumps",
      slope_jump_0, "at t=0 and", slope_jump_tm, "at t=tau_plat.")

## 2. Fourier Transform $\hat{\Gamma}(\omega)$

We compute $\hat{\Gamma}(\omega) = \int_{-\infty}^{\infty} \Gamma(t)\,e^{-i\omega t}\,dt$ by splitting
into three integrals. Since $\Gamma(t)$ is real, we can decompose into real and imaginary parts:
$\hat{\Gamma}(\omega) = \hat{\Gamma}_R(\omega) - i\hat{\Gamma}_I(\omega)$ where

$$\hat{\Gamma}_R(\omega) = \int_{-\infty}^{\infty} \Gamma(t)\cos(\omega t)\,dt, \qquad
\hat{\Gamma}_I(\omega) = \int_{-\infty}^{\infty} \Gamma(t)\sin(\omega t)\,dt$$

We compute each by splitting into rise, plateau, and decay regions.

In [ ]:
# Real part: int Gamma(t) cos(wt) dt
# Rise: int_{-inf}^{0} amax^2 e^{2t/tau_e} cos(wt) dt
I1_R = integrate(amax**2 * exp(2 * t / tau_e) * cos(w * t), (t, -oo, 0))
I1_R = simplify(I1_R)
print("I1_R (rise, cosine):")
display(Math(rf"I_{{1,R}} = {latex(I1_R)}"))

In [ ]:
# Plateau: int_{0}^{tau_plat} amax^2 cos(wt) dt
I2_R = integrate(amax**2 * cos(w * t), (t, 0, tau_p))
I2_R = simplify(I2_R)
print("I2_R (plateau, cosine):")
display(Math(rf"I_{{2,R}} = {latex(I2_R)}"))

In [ ]:
# Decay: int_{tau_plat}^{inf} amax^2 e^{-2(t-tau_plat)/tau_d} cos(wt) dt
I3_R = integrate(amax**2 * exp(-2 * (t - tau_p) / tau_d) * cos(w * t), (t, tau_p, oo))
I3_R = simplify(I3_R)
print("I3_R (decay, cosine):")
display(Math(rf"I_{{3,R}} = {latex(I3_R)}"))

In [ ]:
# Imaginary part: int Gamma(t) sin(wt) dt
I1_I = integrate(amax**2 * exp(2 * t / tau_e) * sin(w * t), (t, -oo, 0))
I2_I = integrate(amax**2 * sin(w * t), (t, 0, tau_p))
I3_I = integrate(amax**2 * exp(-2 * (t - tau_p) / tau_d) * sin(w * t), (t, tau_p, oo))

I1_I, I2_I, I3_I = simplify(I1_I), simplify(I2_I), simplify(I3_I)

Gamma_hat_R = simplify(I1_R + I2_R + I3_R)
Gamma_hat_I = simplify(I1_I + I2_I + I3_I)

print("Real part of Gamma_hat:")
display(Math(rf"\hat{{\Gamma}}_R(\omega) = {latex(Gamma_hat_R)}"))
print("\nImaginary part of Gamma_hat:")
display(Math(rf"\hat{{\Gamma}}_I(\omega) = {latex(Gamma_hat_I)}"))

# Full complex FT: Gamma_hat = Gamma_hat_R - i * Gamma_hat_I
Gamma_hat = Gamma_hat_R - sp.I * Gamma_hat_I
print("\nFull Gamma_hat(omega) = Gamma_hat_R - i * Gamma_hat_I")

### Symmetric case ($\tau_{\rm em} = \tau_{\rm dec} \equiv \tau_s$)

When the emergence and decay timescales are equal, the envelope is symmetric
about $t = \tau_{\rm plat}/2$, and the Fourier transform simplifies.

In [ ]:
tau_s = sp.Symbol('tau_s', positive=True)
Gamma_hat_R_sym = simplify(Gamma_hat_R.subs([(tau_e, tau_s), (tau_d, tau_s)]))
Gamma_hat_I_sym = simplify(Gamma_hat_I.subs([(tau_e, tau_s), (tau_d, tau_s)]))

print("Symmetric case (tau_em = tau_dec = tau_s):")
print("Real part:")
display(Math(rf"\hat{{\Gamma}}_R = {latex(Gamma_hat_R_sym)}"))
print("Imaginary part:")
display(Math(rf"\hat{{\Gamma}}_I = {latex(Gamma_hat_I_sym)}"))

### Check: symmetric case is a phase-shifted real transform

For $\tau_{\rm em} = \tau_{\rm dec}$ the envelope is even about $t = \tau_{\rm plat}/2$, so by the shift theorem
$\hat{\Gamma}(\omega)\,e^{i\omega\tau_{\rm plat}/2}$ must be purely real, i.e.
$\hat{\Gamma}_R\sin(\omega\tau_{\rm plat}/2) - \hat{\Gamma}_I\cos(\omega\tau_{\rm plat}/2) = 0$.

In [ ]:
im_centered = simplify(Gamma_hat_R_sym * sin(w * tau_p / 2) - Gamma_hat_I_sym * cos(w * tau_p / 2))
assert im_centered == 0, f"Im[Gamma_hat e^(i w tau_plat/2)] = {im_centered} != 0 in the symmetric case"
print("Verified: symmetric-case Gamma_hat(omega) * exp(i omega tau_plat/2) is real.")

In [ ]:
# DC limit: Gamma_hat(0) = area under Gamma
# By direct integration:
#   Rise tail:  int_{-inf}^{0} amax^2 e^{2t/tau_e} dt = amax^2 * tau_e / 2
#   Plateau:    int_{0}^{tau_plat} amax^2 dt = amax^2 * tau_plat
#   Decay tail: int_{tau_plat}^{inf} amax^2 e^{-2(t-tau_plat)/tau_d} dt = amax^2 * tau_d / 2
dc_symbolic = amax**2 * (tau_e / 2 + tau_p + tau_d / 2)
print(f"Gamma_hat(0) = area under Gamma = {dc_symbolic}")

# Verify: evaluate Gamma_hat_R at omega -> 0 (imaginary part vanishes at omega=0)
dc_limit_R = sp.limit(Gamma_hat_R, w, 0)
dc_limit_R = simplify(dc_limit_R)
print(f"Symbolic limit of Re[Gamma_hat] as omega->0: {dc_limit_R}")
assert simplify(dc_limit_R - dc_symbolic) == 0
print("Verified: DC limit matches expected area under Gamma.")

### Energy spectral density $|\hat{\Gamma}(\omega)|^2$

For the Wiener-Khinchin theorem, we need $|\hat{\Gamma}(\omega)|^2 = \hat{\Gamma}(\omega) \cdot \hat{\Gamma}^*(\omega)$.
Since $\Gamma(t)$ is real, $\hat{\Gamma}^*(\omega) = \hat{\Gamma}(-\omega)$.

In [ ]:
# |Gamma_hat|^2 = Gamma_hat_R^2 + Gamma_hat_I^2 (real quantity)
ESD = simplify(Gamma_hat_R**2 + Gamma_hat_I**2)
print("|Gamma_hat(omega)|^2 (energy spectral density):")
display(Math(rf"|\hat{{\Gamma}}(\omega)|^2 = {latex(ESD)}"))

### Check: $\hat{\Gamma}^*(\omega) = \hat{\Gamma}(-\omega)$

Because $\Gamma(t)$ is real, $\hat{\Gamma}_R$ must be even and $\hat{\Gamma}_I$ odd in $\omega$.
In particular $\hat{\Gamma}_I \to 0$ as $\omega \to 0$ (the DC value is real), and
$|\hat{\Gamma}(0)|^2 = \big(\int\Gamma\,dt\big)^2$.

In [ ]:
assert simplify(Gamma_hat_R.subs(w, -w) - Gamma_hat_R) == 0, "Gamma_hat_R is not even in omega"
assert simplify(Gamma_hat_I.subs(w, -w) + Gamma_hat_I) == 0, "Gamma_hat_I is not odd in omega"
assert sp.limit(Gamma_hat_I, w, 0) == 0, "Gamma_hat_I does not vanish at omega = 0"
assert simplify(sp.limit(ESD, w, 0) - dc_symbolic**2) == 0, "|Gamma_hat(0)|^2 != (area)^2"
print("Verified: Gamma_hat_R even, Gamma_hat_I odd, Gamma_hat_I(0) = 0, |Gamma_hat(0)|^2 = area^2.")

### Numerical verification of $\hat{\Gamma}(\omega)$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 12


def Gamma_exp_numpy(t_arr, tau_em_n, tau_max_n, tau_dec_n, amax_n):
    """Squared exponential envelope: Gamma(t) = alpha_exp(t)^2."""
    result = np.zeros_like(t_arr, dtype=float)
    rise = t_arr < 0
    plateau = (t_arr >= 0) & (t_arr <= tau_max_n)
    decay = t_arr > tau_max_n
    result[rise] = amax_n**2 * np.exp(2 * t_arr[rise] / tau_em_n)
    result[plateau] = amax_n**2
    result[decay] = amax_n**2 * np.exp(-2 * (t_arr[decay] - tau_max_n) / tau_dec_n)
    return result


def Gamma_hat_numerical(w_arr, t_arr, Gamma_arr):
    """Direct numerical FT for validation."""
    result = np.empty(len(w_arr), dtype=complex)
    for i, wv in enumerate(w_arr):
        integrand = Gamma_arr * np.exp(-1j * wv * t_arr)
        result[i] = np.trapezoid(integrand, t_arr)
    return result

In [ ]:
# Test parameters
tau_em_n, tau_max_n, tau_dec_n, amax_n = 3.0, 15.0, 3.0, 0.1

# Build lambdified analytic Gamma_hat (real and imaginary parts)
Gamma_hat_R_func = sp.lambdify((w, tau_e, tau_p, tau_d, amax), Gamma_hat_R, 'numpy')
Gamma_hat_I_func = sp.lambdify((w, tau_e, tau_p, tau_d, amax), Gamma_hat_I, 'numpy')

def Gamma_hat_func(w_arr, tau_em_n, tau_max_n, tau_dec_n, amax_n):
    """Analytic Gamma_hat as complex array."""
    R = Gamma_hat_R_func(w_arr, tau_em_n, tau_max_n, tau_dec_n, amax_n)
    I = Gamma_hat_I_func(w_arr, tau_em_n, tau_max_n, tau_dec_n, amax_n)
    return R - 1j * I

# Wide time grid (truncate tails at ~10 e-folding times)
t_margin = 10 * max(tau_em_n, tau_dec_n)
t_arr = np.linspace(-t_margin, tau_max_n + t_margin, 50000)
Gamma_arr = Gamma_exp_numpy(t_arr, tau_em_n, tau_max_n, tau_dec_n, amax_n)

# Evaluate
w_arr = np.linspace(0.01, 5.0, 300)
Gh_analytic = Gamma_hat_func(w_arr, tau_em_n, tau_max_n, tau_dec_n, amax_n)
Gh_numerical = Gamma_hat_numerical(w_arr, t_arr, Gamma_arr)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(w_arr, np.abs(Gh_analytic)**2, 'C0', lw=2, label='Analytic')
axes[0].plot(w_arr, np.abs(Gh_numerical)**2, 'C1o', ms=3, alpha=0.5, label='Numerical')
axes[0].set_xlabel(r'$\omega$ [rad/day]')
axes[0].set_ylabel(r'$|\hat{\Gamma}(\omega)|^2$')
axes[0].set_title(r'Energy Spectral Density')
axes[0].set_yscale('log')
axes[0].legend()

rel_err = np.abs(Gh_analytic - Gh_numerical) / np.abs(Gh_numerical)
axes[1].semilogy(w_arr, rel_err, 'C2', lw=1.5)
axes[1].set_xlabel(r'$\omega$ [rad/day]')
axes[1].set_ylabel('Relative error')
axes[1].set_title('Analytic vs Numerical')
axes[1].axhline(1e-3, color='gray', ls='--', lw=0.8)

plt.tight_layout()
plt.show()

print(f"Max relative error: {np.max(rel_err):.2e}")

# Analytic vs direct numerical FT (dashed 1e-3 line in the right panel). The residual
# (~2e-4) comes from the trapezoid rule at the kinks of Gamma and near the zeros of the
# centred real transform; any wrong term in Gamma_hat gives an O(1) error.
assert np.max(rel_err) < 1e-3, f"analytic Gamma_hat disagrees with numerical FT: {np.max(rel_err):.2e}"

### Check: asymmetric timescales

The test above uses $\tau_{\rm em} = \tau_{\rm dec}$, which cannot catch a $\tau_{\rm em} \leftrightarrow \tau_{\rm dec}$
mix-up. Repeat it for $(\tau_{\rm em}, \tau_{\rm dec}) = (1, 5)$ and $(5, 1)$ d, and confirm that the
numerical envelope `Gamma_exp_numpy` used for the validation is $\alpha_{\max}^2$ times the paper's $\Gamma(t)$ (checked in Section 1).

In [ ]:
Gamma_paper_func = sp.lambdify((t, tau_e, tau_p, tau_d, amax), Gamma_paper, 'numpy')

for te_n, td_n in [(1.0, 5.0), (5.0, 1.0)]:
    t_margin_a = 10 * max(te_n, td_n)
    t_a = np.linspace(-t_margin_a, tau_max_n + t_margin_a, 50000)
    G_a = Gamma_exp_numpy(t_a, te_n, tau_max_n, td_n, amax_n)
    assert np.allclose(G_a, amax_n**2 * Gamma_paper_func(t_a, te_n, tau_max_n, td_n, amax_n), rtol=1e-12, atol=0), \
        "Gamma_exp_numpy differs from alpha_max^2 times the paper's Gamma(t)"

    Gh_num_a = Gamma_hat_numerical(w_arr, t_a, G_a)
    Gh_ana_a = Gamma_hat_func(w_arr, te_n, tau_max_n, td_n, amax_n)
    rel_err_a = np.max(np.abs(Gh_ana_a - Gh_num_a) / np.abs(Gh_num_a))
    print(f"tau_em={te_n}, tau_dec={td_n}: max relative error = {rel_err_a:.2e}")
    # observed ~3.5e-6 (trapezoid-rule error); tolerance leaves ~30x margin
    assert rel_err_a < 1e-4, f"analytic Gamma_hat disagrees with numerical FT ({rel_err_a:.2e})"

### Check: $\hat{\Gamma}_R$ and $\hat{\Gamma}_I$ against the paper (`eq:Gamma_hat_exp_R`, `eq:Gamma_hat_exp_I`)

Take both equations from the equation registry (`equations.py`), where they are transcribed term by term
exactly as typeset in `ms.tex`. The paper uses
$\hat{f}(\omega) = \int f(t)\,e^{-i\omega t}\,dt$ and $\hat{\Gamma} = \hat{\Gamma}_R - i\hat{\Gamma}_I$, so
$\hat{\Gamma}_I = \int\Gamma(t)\sin(\omega t)\,dt$, the same convention as above, and since the paper's $\Gamma$ is
normalized, its transforms carry no $\alpha_{\max}^2$ factor: each derived term is divided by $\alpha_{\max}^2$
before the comparison. The derived expressions were validated against direct
numerical integration above. Each term is compared with its piece integral (rise $I_1$, plateau $I_2$,
decay $I_3$) so that a failure names the offending term.

In [ ]:
# eq:Gamma_hat_exp_R and eq:Gamma_hat_exp_I as typeset, from the equation registry (equations.py),
# with their terms grouped by envelope piece
Gamma_hat_R_paper = eqs.get("eq:Gamma_hat_exp_R").expr
Gamma_hat_I_paper = eqs.get("eq:Gamma_hat_exp_I").expr
paper_R_terms, paper_I_terms = eqs.GAMMA_HAT_EXP_R_TERMS, eqs.GAMMA_HAT_EXP_I_TERMS
assert simplify(sum(paper_R_terms.values()) - Gamma_hat_R_paper) == 0
assert simplify(sum(paper_I_terms.values()) - Gamma_hat_I_paper) == 0

derived_R_terms = {'rise': I1_R, 'plateau': I2_R, 'decay': I3_R}
derived_I_terms = {'rise': I1_I, 'plateau': I2_I, 'decay': I3_I}
term_diffs = {}
for part, paper_terms, derived_terms in [('R', paper_R_terms, derived_R_terms),
                                         ('I', paper_I_terms, derived_I_terms)]:
    for piece in paper_terms:
        d = simplify(derived_terms[piece] / amax**2 - paper_terms[piece])
        if d != 0:
            term_diffs[f"Gamma_hat_{part} {piece} term"] = d
            display(Math(rf"\text{{{part}, {piece}: derived}} - \text{{paper}} = {latex(d)}"))

diff_R = simplify(Gamma_hat_R / amax**2 - Gamma_hat_R_paper)
diff_I = simplify(Gamma_hat_I / amax**2 - Gamma_hat_I_paper)

# Independent numerical evidence: paper expressions vs the direct numerical FT
# (tau_em=5, tau_dec=1 grid from the previous cell)
_RpI = sp.lambdify((w, tau_e, tau_p, tau_d, amax), (Gamma_hat_R_paper, Gamma_hat_I_paper), 'numpy')
_Rp, _Ip = _RpI(w_arr, te_n, tau_max_n, td_n, amax_n)
rel_err_paper = np.max(np.abs(amax_n**2 * (_Rp - 1j * _Ip) - Gh_num_a) / np.abs(Gh_num_a))
print(f"Paper eqs vs direct numerical FT (tau_em={te_n}, tau_dec={td_n}): max relative error = {rel_err_paper:.2e}")

assert diff_R == 0, (f"eq:Gamma_hat_exp_R disagrees with the derivation: derived / alpha_max^2 - paper = {diff_R}; "
                     f"per-term differences: {term_diffs}")
assert diff_I == 0, (f"eq:Gamma_hat_exp_I disagrees with the derivation: derived / alpha_max^2 - paper = {diff_I}; "
                     f"per-term differences: {term_diffs}")
print("Verified: eq:Gamma_hat_exp_R and eq:Gamma_hat_exp_I match the derived Gamma_hat_R, Gamma_hat_I.")

## 3. Autocorrelation $R_\Gamma(\tau)$ via Direct Convolution

We compute $R_\Gamma(\tau) = \int_{-\infty}^{\infty} \Gamma(s)\,\Gamma(s+\tau)\,ds$ analytically
by splitting the integral into sub-intervals where both $\Gamma(s)$ and $\Gamma(s+\tau)$ have
known exponential or constant forms.

For $\tau \geq 0$, the regions of $\Gamma(s)$ are:
- Rise: $s < 0$, $\Gamma(s) = \alpha_{\max}^2 e^{2s/\tau_{\rm em}}$
- Plateau: $0 \leq s \leq \tau_{\rm plat}$, $\Gamma(s) = \alpha_{\max}^2$
- Decay: $s > \tau_{\rm plat}$, $\Gamma(s) = \alpha_{\max}^2 e^{-2(s-\tau_{\rm plat})/\tau_{\rm dec}}$

And the regions of $\Gamma(s+\tau)$ are:
- Rise: $s < -\tau$
- Plateau: $-\tau \leq s \leq \tau_{\rm plat} - \tau$
- Decay: $s > \tau_{\rm plat} - \tau$

We consider two cases: $0 \leq \tau \leq \tau_{\rm plat}$ and $\tau > \tau_{\rm plat}$.

### Case 1: $0 \leq \tau \leq \tau_{\rm plat}$

Breakpoints for $s$: $-\tau$, $0$, $\tau_{\rm plat} - \tau$, $\tau_{\rm plat}$.

Sub-intervals and $(\Gamma(s),\;\Gamma(s+\tau))$ regions:
1. $(-\infty,\,-\tau)$: rise $\times$ rise
2. $(-\tau,\,0)$: rise $\times$ plateau
3. $(0,\,\tau_{\rm plat}-\tau)$: plateau $\times$ plateau
4. $(\tau_{\rm plat}-\tau,\,\tau_{\rm plat})$: plateau $\times$ decay
5. $(\tau_{\rm plat},\,\infty)$: decay $\times$ decay

In [ ]:
a2 = amax**2

# Gamma(s) pieces
Gs_rise = a2 * exp(2 * s / tau_e)
Gs_plat = a2
Gs_decay = a2 * exp(-2 * (s - tau_p) / tau_d)

# Gamma(s+T) pieces
GsT_rise = a2 * exp(2 * (s + T) / tau_e)
GsT_plat = a2
GsT_decay = a2 * exp(-2 * (s + T - tau_p) / tau_d)

### Check: convolution pieces

The pieces $\Gamma(s)$ and $\Gamma(s+\tau)$ above must be $\alpha_{\max}^2$ times the branches of the paper's $\Gamma(t)$
(Section 1 check) evaluated at $t = s$ and $t = s + \tau$.

In [ ]:
for mine, branch in [(Gs_rise, G_rise_paper), (Gs_plat, G_plat_paper), (Gs_decay, G_dec_paper)]:
    assert simplify(mine - a2 * branch.subs(t, s)) == 0, f"{mine} != paper branch {branch} at t=s"
for mine, branch in [(GsT_rise, G_rise_paper), (GsT_plat, G_plat_paper), (GsT_decay, G_dec_paper)]:
    assert simplify(mine - a2 * branch.subs(t, s + T)) == 0, f"{mine} != paper branch {branch} at t=s+tau"
print("Verified: Gamma(s), Gamma(s+tau) pieces match alpha_max^2 times the paper's Gamma(t).")

In [ ]:
# Sub-interval 1: (-inf, -T): rise x rise
J1 = integrate(Gs_rise * GsT_rise, (s, -oo, -T))
J1 = simplify(J1)
print("J1 (rise x rise):")
display(Math(rf"J_1 = {latex(J1)}"))

In [ ]:
# Sub-interval 2: (-T, 0): rise x plateau
J2 = integrate(Gs_rise * GsT_plat, (s, -T, 0))
J2 = simplify(J2)
print("J2 (rise x plateau):")
display(Math(rf"J_2 = {latex(J2)}"))

In [ ]:
# Sub-interval 3: (0, tau_plat - T): plateau x plateau
J3 = integrate(Gs_plat * GsT_plat, (s, 0, tau_p - T))
J3 = simplify(J3)
print("J3 (plateau x plateau):")
display(Math(rf"J_3 = {latex(J3)}"))

In [ ]:
# Sub-interval 4: (tau_plat - T, tau_plat): plateau x decay
J4 = integrate(Gs_plat * GsT_decay, (s, tau_p - T, tau_p))
J4 = simplify(J4)
print("J4 (plateau x decay):")
display(Math(rf"J_4 = {latex(J4)}"))

In [ ]:
# Sub-interval 5: (tau_plat, inf): decay x decay
J5 = integrate(Gs_decay * GsT_decay, (s, tau_p, oo))
J5 = simplify(J5)
print("J5 (decay x decay):")
display(Math(rf"J_5 = {latex(J5)}"))

In [ ]:
R_case1 = simplify(J1 + J2 + J3 + J4 + J5)
print("R_Gamma for 0 <= tau <= tau_plat:")
display(Math(rf"R_\Gamma(\tau) = {latex(R_case1)}"))

### Case 2: $\tau > \tau_{\rm plat}$

When $\tau > \tau_{\rm plat}$, the shifted plateau $[-\tau,\,\tau_{\rm plat}-\tau]$ lies entirely
in $s < 0$ (the rise region of $\Gamma(s)$). The plateau of $\Gamma(s+\tau)$ starts at
$s = -\tau$ and ends at $s = \tau_{\rm plat} - \tau < 0$.

Breakpoints: $-\tau$, $\tau_{\rm plat} - \tau$, $0$, $\tau_{\rm plat}$.

Sub-intervals:
1. $(-\infty,\,-\tau)$: rise $\times$ rise
2. $(-\tau,\,\tau_{\rm plat}-\tau)$: rise $\times$ plateau
3. $(\tau_{\rm plat}-\tau,\,0)$: rise $\times$ decay
4. $(0,\,\tau_{\rm plat})$: plateau $\times$ decay
5. $(\tau_{\rm plat},\,\infty)$: decay $\times$ decay

In [ ]:
# Sub-interval 1: (-inf, -T): rise x rise (same as Case 1)
K1 = J1  # identical expression
print("K1 (rise x rise):")
display(Math(rf"K_1 = {latex(K1)}"))

In [ ]:
# Sub-interval 2: (-T, tau_plat - T): rise x plateau
K2 = integrate(Gs_rise * GsT_plat, (s, -T, tau_p - T))
K2 = simplify(K2)
print("K2 (rise x plateau):")
display(Math(rf"K_2 = {latex(K2)}"))

In [ ]:
# Sub-interval 3: (tau_plat - T, 0): rise x decay
K3 = integrate(Gs_rise * GsT_decay, (s, tau_p - T, 0))
K3 = simplify(K3)
print("K3 (rise x decay):")
display(Math(rf"K_3 = {latex(K3)}"))

In [ ]:
# Sub-interval 4: (0, tau_plat): plateau x decay
K4 = integrate(Gs_plat * GsT_decay, (s, 0, tau_p))
K4 = simplify(K4)
print("K4 (plateau x decay):")
display(Math(rf"K_4 = {latex(K4)}"))

In [ ]:
# Sub-interval 5: (tau_plat, inf): decay x decay (same as Case 1)
K5 = J5  # identical expression
print("K5 (decay x decay):")
display(Math(rf"K_5 = {latex(K5)}"))

In [ ]:
R_case2 = simplify(K1 + K2 + K3 + K4 + K5)
print("R_Gamma for tau > tau_plat:")
display(Math(rf"R_\Gamma(\tau) = {latex(R_case2)}"))

### Continuity check at $\tau = \tau_{\rm plat}$

In [ ]:
R1_at_tm = simplify(R_case1.subs(T, tau_p))
R2_at_tm = simplify(R_case2.subs(T, tau_p))

print("R_case1(tau_plat):")
display(R1_at_tm)
print("R_case2(tau_plat):")
display(R2_at_tm)
print(f"\nMatch: {simplify(R1_at_tm - R2_at_tm) == 0}")

assert simplify(R1_at_tm - R2_at_tm) == 0, "R_Gamma is discontinuous at tau = tau_plat"

### Value at $\tau = 0$: $R_\Gamma(0)$

In [ ]:
R_at_0 = simplify(R_case1.subs(T, 0))
print("R_Gamma(0):")
display(Math(rf"R_\Gamma(0) = {latex(R_at_0)}"))

# Expected: int Gamma^2 ds = amax^4 * (tau_e/4 + tau_plat + tau_d/4)
expected_R0 = amax**4 * (tau_e / 4 + tau_p + tau_d / 4)
print(f"\nExpected: alpha_max^4 * (tau_em/4 + tau_plat + tau_dec/4) = {expected_R0}")
print(f"Match: {simplify(R_at_0 - expected_R0) == 0}")

assert simplify(R_at_0 - expected_R0) == 0, f"R_Gamma(0) = {R_at_0} != {expected_R0}"

### Symmetric case ($\tau_{\rm em} = \tau_{\rm dec} = \tau_s$)

Simplify the expressions for equal emergence and decay timescales.

In [ ]:
R_case1_sym = simplify(R_case1.subs([(tau_e, tau_s), (tau_d, tau_s)]))
R_case2_sym = simplify(R_case2.subs([(tau_e, tau_s), (tau_d, tau_s)]))

print("Symmetric R_Gamma, Case 1 (0 <= tau <= tau_plat):")
display(Math(rf"R_\Gamma(\tau) = {latex(R_case1_sym)}"))

print("\nSymmetric R_Gamma, Case 2 (tau > tau_plat):")
display(Math(rf"R_\Gamma(\tau) = {latex(R_case2_sym)}"))

### Check: degenerate branch and $\tau_{\rm em} \leftrightarrow \tau_{\rm dec}$ invariance

(i) SymPy returns Case 2 as a `Piecewise` with a separate $\tau_{\rm em} = \tau_{\rm dec}$ branch; it must be the
$\tau_{\rm dec} \to \tau_{\rm em}$ limit of the generic branch and agree with the symmetric expression above.
(ii) Time-reversing $\Gamma$ about the plateau centre swaps $\tau_{\rm em}$ and $\tau_{\rm dec}$ and maps
$R_\Gamma(\tau) \to R_\Gamma(-\tau) = R_\Gamma(\tau)$, so $R_\Gamma$ must be invariant under the swap.

In [ ]:
assert isinstance(R_case2, sp.Piecewise) and len(R_case2.args) == 2
(R2_generic, cond_generic), (R2_equal, cond_equal) = R_case2.args
assert cond_generic in (sp.Ne(tau_d, tau_e), sp.Ne(tau_e, tau_d)) and cond_equal == True

eps = sp.Symbol('epsilon', positive=True)
lim_generic = sp.limit(R2_generic.subs(tau_d, tau_e + eps), eps, 0)
assert simplify(lim_generic - R2_equal.subs(tau_d, tau_e)) == 0, "degenerate branch != limit of generic branch"
assert simplify(R2_equal.subs({tau_e: tau_s, tau_d: tau_s}) - R_case2_sym) == 0

swap = {tau_e: tau_d, tau_d: tau_e}
assert simplify(R_case1 - R_case1.subs(swap, simultaneous=True)) == 0, "R_case1 not symmetric in tau_em <-> tau_dec"
assert simplify(R2_generic - R2_generic.subs(swap, simultaneous=True)) == 0, "R_case2 not symmetric in tau_em <-> tau_dec"
print("Verified: degenerate branch is the tau_dec -> tau_em limit; R_Gamma is invariant under tau_em <-> tau_dec.")

### Numerical verification of $R_\Gamma(\tau)$

In [ ]:
def R_Gamma_numerical(tau_arr, t_grid, Gamma_grid, tau_em_n, tau_max_n, tau_dec_n, amax_n):
    """R_Gamma(tau) via direct numerical convolution."""
    R = np.empty_like(tau_arr)
    for i, tau_val in enumerate(tau_arr):
        Gamma_shifted = Gamma_exp_numpy(t_grid + tau_val, tau_em_n, tau_max_n, tau_dec_n, amax_n)
        R[i] = np.trapezoid(Gamma_grid * Gamma_shifted, t_grid)
    return R


# Lambdify analytic expressions
R_case1_func = sp.lambdify((T, tau_e, tau_p, tau_d, amax), R_case1, 'numpy')
R_case2_func = sp.lambdify((T, tau_e, tau_p, tau_d, amax), R_case2, 'numpy')


def R_Gamma_analytic(tau_arr, tau_em_n, tau_max_n, tau_dec_n, amax_n):
    """Evaluate piecewise analytic R_Gamma."""
    R = np.empty_like(tau_arr)
    case1 = tau_arr <= tau_max_n
    case2 = tau_arr > tau_max_n
    if np.any(case1):
        R[case1] = R_case1_func(tau_arr[case1], tau_em_n, tau_max_n, tau_dec_n, amax_n)
    if np.any(case2):
        R[case2] = R_case2_func(tau_arr[case2], tau_em_n, tau_max_n, tau_dec_n, amax_n)
    return R

In [ ]:
tau_test = np.linspace(0, tau_max_n + 6 * max(tau_em_n, tau_dec_n), 300)

R_ana = R_Gamma_analytic(tau_test, tau_em_n, tau_max_n, tau_dec_n, amax_n)
R_num = R_Gamma_numerical(tau_test, t_arr, Gamma_arr, tau_em_n, tau_max_n, tau_dec_n, amax_n)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(tau_test, R_ana / R_ana[0], 'C0', lw=2, label='Analytic')
axes[0].plot(tau_test, R_num / R_num[0], 'C1o', ms=3, alpha=0.5, label='Numerical')
axes[0].set_xlabel(r'$\tau$ [days]')
axes[0].set_ylabel(r'$R_\Gamma(\tau) / R_\Gamma(0)$')
axes[0].set_title(r'Normalized $R_\Gamma(\tau)$')
axes[0].axvline(tau_max_n, color='gray', ls=':', lw=0.8, label=r'$\tau_{\rm plat}$')
axes[0].legend()

rel_err_R = np.abs(R_ana - R_num) / np.abs(R_num[0])
axes[1].semilogy(tau_test, rel_err_R, 'C2', lw=1.5)
axes[1].set_xlabel(r'$\tau$ [days]')
axes[1].set_ylabel('Absolute error / R(0)')
axes[1].set_title('Analytic vs Numerical')
axes[1].axhline(1e-6, color='gray', ls='--', lw=0.8)

plt.tight_layout()
plt.show()

print(f"Max relative error: {np.max(rel_err_R):.2e}")

# Analytic vs direct numerical convolution (dashed 1e-6 line); observed ~1e-8
assert np.max(rel_err_R) < 1e-6, f"analytic R_Gamma disagrees with numerical convolution: {np.max(rel_err_R):.2e}"

### Check: asymmetric $R_\Gamma$ and Wiener–Khinchin consistency (`eq:wk`)

Repeat the convolution test for $(\tau_{\rm em}, \tau_{\rm dec}) = (1, 5)$ and $(5, 1)$ d, and check that the two
closed forms are mutually consistent via the Wiener–Khinchin theorem (`eq:wk`): for real, even $R_\Gamma$,
$|\hat{\Gamma}(\omega)|^2 = 2\int_0^\infty R_\Gamma(\tau)\cos(\omega\tau)\,d\tau$. This ties the Section 2 result
($|\hat{\Gamma}|^2$) to the Section 3 result ($R_\Gamma$).

In [ ]:
ESD_func = sp.lambdify((w, tau_e, tau_p, tau_d, amax), ESD, 'numpy')
w_wk = np.linspace(0.05, 3.0, 60)

for te_n, td_n in [(1.0, 5.0), (5.0, 1.0)]:
    t_margin_a = 10 * max(te_n, td_n)
    t_a = np.linspace(-t_margin_a, tau_max_n + t_margin_a, 50000)
    G_a = Gamma_exp_numpy(t_a, te_n, tau_max_n, td_n, amax_n)
    tau_a = np.linspace(0, tau_max_n + 6 * max(te_n, td_n), 300)
    R_ana_a = R_Gamma_analytic(tau_a, te_n, tau_max_n, td_n, amax_n)
    R_num_a = R_Gamma_numerical(tau_a, t_a, G_a, te_n, tau_max_n, td_n, amax_n)
    err_R_a = np.max(np.abs(R_ana_a - R_num_a)) / R_ana_a[0]

    # Wiener-Khinchin: cosine transform of the analytic R_Gamma vs analytic |Gamma_hat|^2
    tau_g = np.linspace(0, tau_max_n + 30 * max(te_n, td_n), 20001)
    R_g = R_Gamma_analytic(tau_g, te_n, tau_max_n, td_n, amax_n)
    S_from_R = np.array([2 * np.trapezoid(R_g * np.cos(wv * tau_g), tau_g) for wv in w_wk])
    S_ana = ESD_func(w_wk, te_n, tau_max_n, td_n, amax_n)
    S_0 = (amax_n**2 * (te_n / 2 + tau_max_n + td_n / 2))**2   # |Gamma_hat(0)|^2
    err_wk = np.max(np.abs(S_from_R - S_ana)) / S_0
    print(f"tau_em={te_n}, tau_dec={td_n}: R_Gamma err/R(0) = {err_R_a:.2e}, WK err/|Gamma_hat(0)|^2 = {err_wk:.2e}")
    assert err_R_a < 1e-6, f"analytic R_Gamma disagrees with numerical convolution ({err_R_a:.2e})"
    # observed ~1e-13 (R_Gamma is smooth and decays exponentially, so the quadrature is very accurate)
    assert err_wk < 1e-8, f"|Gamma_hat|^2 and R_Gamma violate Wiener-Khinchin ({err_wk:.2e})"

## 4. Derivative Discontinuities (Kinks)

The piecewise exponential $\alpha(t)$ is $C^0$ (continuous but with kinks at $t=0$ and $t=\tau_{\rm plat}$).
These kinks propagate into $R_\Gamma(\tau)$, producing derivative discontinuities.
We verify this by comparing derivatives of the analytic $R_\Gamma(\tau)$ in the two cases.

In [ ]:
# Derivative of R_Gamma in each case
dR_case1 = sp.diff(R_case1, T)
dR_case2 = sp.diff(R_case2, T)

# Evaluate derivatives at tau = tau_plat from both sides
dR1_at_tm = simplify(dR_case1.subs(T, tau_p))
dR2_at_tm = simplify(dR_case2.subs(T, tau_p))

print("dR/dtau at tau = tau_plat:")
print(f"  From Case 1 (left):  {dR1_at_tm}")
print(f"  From Case 2 (right): {dR2_at_tm}")

jump = simplify(dR2_at_tm - dR1_at_tm)
print(f"  Derivative jump: {jump}")

if jump != 0:
    print("\n  => R_Gamma has a kink at tau = tau_plat (derivative discontinuity).")
else:
    print("\n  => R_Gamma is C^1 at tau = tau_plat.")

### Check: smoothness class of $R_\Gamma$

The derivative jump printed above is zero: $R'_\Gamma$ is continuous at $\tau = \tau_{\rm plat}$. Because $\Gamma$ itself is
continuous (only $\Gamma'$ jumps), $R_\Gamma = \Gamma \star \Gamma$ is in fact $C^2$; the first discontinuous
derivative is $R'''_\Gamma$, which jumps by $4\alpha_{\max}^4/(\tau_{\rm em}\tau_{\rm dec})$ at $\tau = \tau_{\rm plat}$.
At $\tau = 0$, $R'_\Gamma(0^+) = 0$ (no cusp), while $R'''_\Gamma(0^+) = 2\alpha_{\max}^4(\tau_{\rm em}^{-2} + \tau_{\rm dec}^{-2}) \neq 0$
(and $R'''_\Gamma$ is odd, so it jumps at 0). We assert this structure (both `Piecewise` branches of Case 2).

In [ ]:
assert jump == 0, f"dR/dtau jumps by {jump} at tau = tau_plat"

def _jump_at_tau_max(k):
    # (generic tau_em != tau_dec branch, degenerate tau_em == tau_dec branch)
    d1 = sp.diff(R_case1, T, k).subs(T, tau_p)
    j_generic = simplify(sp.diff(R2_generic, T, k).subs(T, tau_p) - d1)
    j_equal = simplify((sp.diff(R2_equal, T, k).subs(T, tau_p) - d1).subs(tau_d, tau_e))
    return j_generic, j_equal

for k in (1, 2):
    assert _jump_at_tau_max(k) == (0, 0), f"d^{k}R/dtau^{k} jumps at tau = tau_plat: {_jump_at_tau_max(k)}"
j3_generic, j3_equal = _jump_at_tau_max(3)
assert simplify(j3_generic - 4 * amax**4 / (tau_e * tau_d)) == 0, j3_generic
assert simplify(j3_equal - 4 * amax**4 / tau_e**2) == 0, j3_equal

assert simplify(sp.diff(R_case1, T).subs(T, 0)) == 0, "R_Gamma has a cusp at tau = 0"
assert simplify(sp.diff(R_case1, T, 3).subs(T, 0) - 2 * amax**4 * (1 / tau_e**2 + 1 / tau_d**2)) == 0
print("Verified: R_Gamma is C^2 at tau = tau_plat and tau = 0; d^3R/dtau^3 jumps by",
      j3_generic, "at tau_plat.")

In [ ]:
# Visualize the kink via numerical derivative
dtau = tau_test[1] - tau_test[0]
dR_ana = np.gradient(R_ana, dtau)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(tau_test, R_ana / R_ana[0], 'C0', lw=2)
axes[0].set_xlabel(r'$\tau$ [days]')
axes[0].set_ylabel(r'$R_\Gamma(\tau) / R_\Gamma(0)$')
axes[0].set_title(r'$R_\Gamma(\tau)$')
axes[0].axvline(tau_max_n, color='C3', ls=':', lw=1, label=r'$\tau_{\rm plat}$')
axes[0].legend()

axes[1].plot(tau_test, dR_ana, 'C0', lw=2)
axes[1].set_xlabel(r'$\tau$ [days]')
axes[1].set_ylabel(r"$R'_\Gamma(\tau)$")
axes[1].set_title(r"First derivative of $R_\Gamma(\tau)$")
axes[1].axvline(tau_max_n, color='C3', ls=':', lw=1, label=r'$\tau_{\rm plat}$ (kink)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Comparison to Trapezoidal Envelope

Compare $R_\Gamma(\tau)$ for the exponential and trapezoidal envelopes
with matched parameters ($\ell_{\rm spot} = \tau_{\rm plat}$, $\tau_{\rm spot} = \tau_{\rm em} = \tau_{\rm dec}$).

In [ ]:
def Gamma_trapezoid(t_arr, ell_n, tau_s_n, amax_n):
    """Squared trapezoidal envelope for comparison."""
    abs_t = np.abs(t_arr)
    L = ell_n / 2
    S = L + tau_s_n
    result = np.zeros_like(abs_t)
    plateau = abs_t <= L
    ramp = (abs_t > L) & (abs_t <= S)
    result[plateau] = amax_n**2
    result[ramp] = amax_n**2 * ((S - abs_t[ramp]) / tau_s_n)**2
    return result


# Matched parameters
ell_trap = tau_max_n
tau_s_trap = tau_em_n  # = tau_dec_n

# Center the trapezoidal envelope at tau_plat/2 for fair comparison
t_trap = t_arr - tau_max_n / 2  # shift so plateau is centered at 0
Gamma_trap_arr = Gamma_trapezoid(t_trap, ell_trap, tau_s_trap, amax_n)

# R_Gamma for trapezoidal (numerical)
R_trap = np.empty_like(tau_test)
for i, tv in enumerate(tau_test):
    Gt_shifted = Gamma_trapezoid(t_trap + tv, ell_trap, tau_s_trap, amax_n)
    R_trap[i] = np.trapezoid(Gamma_trap_arr * Gt_shifted, t_arr)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Envelopes
axes[0].plot(t_arr, Gamma_arr / amax_n**2, 'C0', lw=2, label='Exponential')
axes[0].plot(t_arr, Gamma_trap_arr / amax_n**2, 'C1', lw=2, ls='--', label='Trapezoidal')
axes[0].set_xlabel('Time [days]')
axes[0].set_ylabel(r'$\Gamma(t) / \alpha_{\max}^2$')
axes[0].set_title(r'Squared Envelopes')
axes[0].legend()
axes[0].set_xlim(-10, tau_max_n + 10)

# R_Gamma
axes[1].plot(tau_test, R_ana / R_ana[0], 'C0', lw=2, label='Exponential')
axes[1].plot(tau_test, R_trap / R_trap[0], 'C1', lw=2, ls='--', label='Trapezoidal')
axes[1].set_xlabel(r'$\tau$ [days]')
axes[1].set_ylabel(r'$R_\Gamma(\tau) / R_\Gamma(0)$')
axes[1].set_title(r'$R_\Gamma(\tau)$')
axes[1].legend()

# Derivatives
dR_exp = np.gradient(R_ana / R_ana[0], dtau)
dR_trap_num = np.gradient(R_trap / R_trap[0], dtau)

axes[2].plot(tau_test, dR_exp, 'C0', lw=2, label='Exponential')
axes[2].plot(tau_test, dR_trap_num, 'C1', lw=2, ls='--', label='Trapezoidal')
axes[2].set_xlabel(r'$\tau$ [days]')
axes[2].set_ylabel(r"$R'_\Gamma(\tau)$")
axes[2].set_title('First derivative')
axes[2].legend()

plt.tight_layout()
plt.show()

print("Key differences:")
print("  - Exponential envelope has infinite support (exponential tails)")
print("  - Trapezoidal envelope has compact support (ell + 2*tau_s)")
print("  - Both have derivative discontinuities in R_Gamma")
print("  - Exponential R_Gamma has a kink at tau = tau_plat")
print("  - Trapezoidal R_Gamma has kinks at tau_s, ell, ell+tau_s")

### Check: support claims

The squared trapezoid has compact support $\ell + 2\tau_s$, so its $R_\Gamma$ vanishes identically for
$\tau \geq \ell + 2\tau_s$, and $R_\Gamma(0) = \alpha_{\max}^4(\ell + 2\tau_s/5)$ (summary table). The exponential
envelope has infinite support, so its $R_\Gamma$ stays strictly positive at all lags.

In [ ]:
far = tau_test >= ell_trap + 2 * tau_s_trap
assert far.any() and np.all(R_trap[far] == 0.0), "trapezoid R_Gamma nonzero beyond ell + 2 tau_s"
# observed relative error ~1e-8 (trapezoid rule on the 50000-point grid)
assert np.isclose(R_trap[0], amax_n**4 * (ell_trap + 2 * tau_s_trap / 5), rtol=1e-6, atol=0), R_trap[0]
assert np.all(R_ana > 0), "exponential R_Gamma is not strictly positive"
print("Verified: trapezoid R_Gamma has compact support and R(0) = a^4 (ell + 2 tau_s/5); exponential R_Gamma > 0.")

## 6. Assembling the Full Single-Spot Kernel

From Eq. (`kernel_single_spot`):
$$k(\tau) = \alpha_{\max}^4\, R_\Gamma(\tau) \left[ |c_0|^2 + 2\sum_{n=1}^{N} |c_n|^2\, \cos(n\omega_0\tau) \right]$$

The Fourier coefficients $c_n$ of the visibility function are the same regardless of
envelope shape (they depend only on $(I, \Phi)$).

In [ ]:
def visibility_fourier_coefficients(inc, phi, n_harmonics=2):
    """Compute Fourier coefficients c_n of Pi(t) = max{cos(beta(t)), 0}.
    From Eq. (cn_general) in the paper.
    """
    a0 = np.cos(inc) * np.sin(phi)
    a1 = np.sin(inc) * np.cos(phi)

    if np.abs(a0) >= a1:
        if a0 > 0:
            theta_vis = np.pi
        else:
            return np.zeros(n_harmonics + 1)
    else:
        theta_vis = np.arccos(-a0 / a1)

    c = np.zeros(n_harmonics + 1)
    c[0] = (a0 * theta_vis + a1 * np.sin(theta_vis)) / np.pi
    c[1] = (a0 * np.sin(theta_vis) +
            a1 * (theta_vis + np.sin(theta_vis) * np.cos(theta_vis)) / 2) / np.pi
    for n in range(2, n_harmonics + 1):
        term1 = a0 * np.sin(n * theta_vis) / n
        term2 = a1 / 2 * (
            np.sin((n - 1) * theta_vis) / (n - 1) +
            np.sin((n + 1) * theta_vis) / (n + 1)
        )
        c[n] = (term1 + term2) / np.pi
    return c


def kernel_single_spot(tau_arr, R_Gamma_arr, cn, omega0, alpha_max):
    """Single-spot kernel (Eq. kernel_single_spot)."""
    cosine_sum = cn[0]**2 + 2 * sum(
        cn[n]**2 * np.cos(n * omega0 * tau_arr)
        for n in range(1, len(cn))
    )
    return alpha_max**4 * R_Gamma_arr * cosine_sum

In [ ]:
# Kernel parameters
P_eq = 5.0        # rotation period [days]
inc = np.pi / 2   # edge-on
phi = np.pi / 4   # 45 deg latitude
omega0 = 2 * np.pi / P_eq
n_harmonics = 2

cn = visibility_fourier_coefficients(inc, phi, n_harmonics)
print(f"Fourier coefficients (I={np.degrees(inc):.0f} deg, Phi={np.degrees(phi):.0f} deg):")
for n, c in enumerate(cn):
    print(f"  c_{n} = {c:.6f}")

# Kernel lag grid
tau_kernel = np.linspace(0, 40, 2000)

# Exponential R_Gamma (analytic)
R_exp_kernel = R_Gamma_analytic(tau_kernel, tau_em_n, tau_max_n, tau_dec_n, amax_n)
R_exp_norm = R_exp_kernel / R_exp_kernel[0]

# Trapezoidal R_Gamma (numerical)
R_trap_kernel = np.empty_like(tau_kernel)
for i, tv in enumerate(tau_kernel):
    Gt_shifted = Gamma_trapezoid(t_trap + tv, ell_trap, tau_s_trap, amax_n)
    R_trap_kernel[i] = np.trapezoid(Gamma_trap_arr * Gt_shifted, t_arr)
R_trap_norm = R_trap_kernel / R_trap_kernel[0]

# Build kernels
k_exp = kernel_single_spot(tau_kernel, R_exp_norm, cn, omega0, amax_n)
k_trap = kernel_single_spot(tau_kernel, R_trap_norm, cn, omega0, amax_n)

# Normalize
k_exp_norm = k_exp / k_exp[0]
k_trap_norm = k_trap / k_trap[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(tau_kernel, k_exp_norm, 'C0', lw=2, label='Exponential')
axes[0].plot(tau_kernel, k_trap_norm, 'C1', lw=2, ls='--', label='Trapezoidal')
axes[0].set_xlabel(r'$\tau$ [days]')
axes[0].set_ylabel(r'$k(\tau) / k(0)$')
axes[0].set_title(f'Normalized kernel ($P_{{eq}}$={P_eq}d, $I$={np.degrees(inc):.0f}$^\circ$, $\Phi$={np.degrees(phi):.0f}$^\circ$)')
axes[0].legend()
axes[0].axhline(0, color='gray', ls='--', lw=0.5)

axes[1].plot(tau_kernel, k_exp_norm, 'C0', lw=2, label='Exponential')
axes[1].plot(tau_kernel, k_trap_norm, 'C1', lw=2, ls='--', label='Trapezoidal')
axes[1].set_xlabel(r'$\tau$ [days]')
axes[1].set_ylabel(r'$k(\tau) / k(0)$')
axes[1].set_title('Zoom: first 3 periods')
axes[1].set_xlim(0, 3 * P_eq)
axes[1].legend()
axes[1].axhline(0, color='gray', ls='--', lw=0.5)

plt.tight_layout()
plt.show()

### Check: visibility coefficients $c_n$ (`eq:cn_general`)

The $c_n$ entering the kernel (`eq:kernel_single_spot`) must be the Fourier coefficients
$c_n = \frac{1}{2\pi}\int_{-\pi}^{\pi}\max\{a_0 + a_1\cos\theta,\,0\}\cos(n\theta)\,d\theta$ with
$a_0 = \cos I\sin\Phi$, $a_1 = \sin I\cos\Phi$. Compare against direct quadrature for the geometry used above
and for partially visible, always visible, and never visible spots.

In [ ]:
theta_q = np.linspace(-np.pi, np.pi, 200001)
for inc_c, phi_c in [(inc, phi), (np.radians(60), np.radians(30)), (np.radians(60), np.radians(-50)),
                     (np.radians(30), np.radians(70)), (np.radians(20), np.radians(-80))]:
    a0_c, a1_c = np.cos(inc_c) * np.sin(phi_c), np.sin(inc_c) * np.cos(phi_c)
    Pi_q = np.maximum(a0_c + a1_c * np.cos(theta_q), 0.0)
    cn_quad = np.array([np.trapezoid(Pi_q * np.cos(n * theta_q), theta_q) / (2 * np.pi)
                        for n in range(n_harmonics + 1)])
    cn_c = visibility_fourier_coefficients(inc_c, phi_c, n_harmonics)
    # observed agreement ~1e-11 (quadrature error at the visibility kinks)
    assert np.allclose(cn_c, cn_quad, rtol=0, atol=1e-8), (np.degrees([inc_c, phi_c]), cn_c, cn_quad)
print("Verified: c_n match direct quadrature of the visibility function.")

## 7. Asymmetric Emergence/Decay

Unlike the trapezoidal envelope (which assumes $\tau_{\rm em} = \tau_{\rm dec}$),
the exponential model naturally supports asymmetric timescales.

In [ ]:
def alpha_exp_numpy(t_arr, tau_em_n, tau_max_n, tau_dec_n, amax_n):
    """Piecewise exponential envelope (Eq. 10)."""
    result = np.zeros_like(t_arr, dtype=float)
    rise = t_arr < 0
    plateau = (t_arr >= 0) & (t_arr <= tau_max_n)
    decay = t_arr > tau_max_n
    result[rise] = amax_n * np.exp(t_arr[rise] / tau_em_n)
    result[plateau] = amax_n
    result[decay] = amax_n * np.exp(-(t_arr[decay] - tau_max_n) / tau_dec_n)
    return result


fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cases = [
    (1.0, 5.0, r'$\tau_{\rm em}=1, \tau_{\rm dec}=5$ (fast rise, slow decay)'),
    (3.0, 3.0, r'$\tau_{\rm em}=3, \tau_{\rm dec}=3$ (symmetric)'),
    (5.0, 1.0, r'$\tau_{\rm em}=5, \tau_{\rm dec}=1$ (slow rise, fast decay)'),
]

for te, td, label in cases:
    alpha_tmp = alpha_exp_numpy(t_arr, te, tau_max_n, td, amax_n)
    axes[0].plot(t_arr, alpha_tmp / amax_n, lw=2, label=label)

    R_tmp = R_Gamma_analytic(tau_kernel, te, tau_max_n, td, amax_n)
    k_tmp = kernel_single_spot(tau_kernel, R_tmp / R_tmp[0], cn, omega0, amax_n)
    axes[1].plot(tau_kernel, k_tmp / k_tmp[0], lw=2, label=label)

axes[0].set_xlabel('Time [days]')
axes[0].set_ylabel(r'$\alpha(t) / \alpha_{\max}$')
axes[0].set_title('Asymmetric exponential envelopes')
axes[0].legend(fontsize=9)
axes[0].set_xlim(-15, tau_max_n + 15)

axes[1].set_xlabel(r'$\tau$ [days]')
axes[1].set_ylabel(r'$k(\tau)/k(0)$')
axes[1].set_title('Resulting kernels')
axes[1].legend(fontsize=9)
axes[1].axhline(0, color='gray', ls='--', lw=0.5)

plt.tight_layout()
plt.show()

print("The kernel k(tau) is always symmetric in tau (valid covariance function),")
print("even when the envelope is asymmetric, because R_Gamma(tau) = R_Gamma(-tau).")

### Check: $R_\Gamma(-\tau) = R_\Gamma(\tau)$ for asymmetric envelopes

Verify the claim printed above for the three $(\tau_{\rm em}, \tau_{\rm dec})$ cases plotted: the direct numerical
convolution at negative lags must equal the analytic $R_\Gamma(|\tau|)$.

In [ ]:
for te_n, td_n, _ in cases:
    t_margin_a = 10 * max(te_n, td_n)
    t_a = np.linspace(-t_margin_a, tau_max_n + t_margin_a, 50000)
    G_a = Gamma_exp_numpy(t_a, te_n, tau_max_n, td_n, amax_n)
    tau_a = np.linspace(0, tau_max_n + 6 * max(te_n, td_n), 300)
    R_pos = R_Gamma_analytic(tau_a, te_n, tau_max_n, td_n, amax_n)
    R_neg = R_Gamma_numerical(-tau_a, t_a, G_a, te_n, tau_max_n, td_n, amax_n)
    err_sym = np.max(np.abs(R_neg - R_pos)) / R_pos[0]
    print(f"tau_em={te_n}, tau_dec={td_n}: max |R(-tau) - R(tau)| / R(0) = {err_sym:.2e}")
    assert err_sym < 1e-6, f"R_Gamma(-tau) != R_Gamma(tau) ({err_sym:.2e})"

## 8. Parameter Sensitivity

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Vary tau_em (= tau_dec for symmetric case)
for ts in [1.0, 3.0, 8.0]:
    R_tmp = R_Gamma_analytic(tau_kernel, ts, tau_max_n, ts, amax_n)
    k_tmp = kernel_single_spot(tau_kernel, R_tmp / R_tmp[0], cn, omega0, amax_n)
    axes[0].plot(tau_kernel, k_tmp / k_tmp[0], lw=2, label=rf'$\tau_s={ts}$')
axes[0].set_title(r'Vary $\tau_s = \tau_{\rm em} = \tau_{\rm dec}$')

# Vary tau_plat
for tm in [5.0, 15.0, 30.0]:
    R_tmp = R_Gamma_analytic(tau_kernel, tau_em_n, tm, tau_dec_n, amax_n)
    k_tmp = kernel_single_spot(tau_kernel, R_tmp / R_tmp[0], cn, omega0, amax_n)
    axes[1].plot(tau_kernel, k_tmp / k_tmp[0], lw=2, label=rf'$\tau_{{\rm plat}}={tm}$')
axes[1].set_title(r'Vary $\tau_{\rm plat}$ (plateau duration)')

# Vary P_eq
for P in [3.0, 5.0, 10.0]:
    w0 = 2 * np.pi / P
    R_tmp = R_Gamma_analytic(tau_kernel, tau_em_n, tau_max_n, tau_dec_n, amax_n)
    k_tmp = kernel_single_spot(tau_kernel, R_tmp / R_tmp[0], cn, w0, amax_n)
    axes[2].plot(tau_kernel, k_tmp / k_tmp[0], lw=2, label=rf'$P_{{eq}}={P}$d')
axes[2].set_title(r'Vary $P_{\rm eq}$')

for ax in axes:
    ax.set_xlabel(r'$\tau$ [days]')
    ax.set_ylabel(r'$k(\tau)/k(0)$')
    ax.legend(fontsize=10)
    ax.axhline(0, color='gray', ls='--', lw=0.5)

plt.tight_layout()
plt.show()

## Summary

The piecewise exponential envelope (Eq. 10) yields a kernel with the same factorized structure:

$$k(\tau) = \alpha_{\max}^4\, R_\Gamma(\tau) \left[ |c_0|^2 + 2\sum_{n=1}^{N} |c_n|^2\, \cos(n\omega_0\tau) \right]$$

**Key results:**

| Property | Trapezoidal | Exponential |
|---|---|---|
| $\Gamma(t) = \alpha^2(t)$ | Piecewise polynomial (parabolic ramps) | Piecewise: $e^{2t/\tau_{\rm em}}$, 1, $e^{-2(t-\tau_{\rm plat})/\tau_{\rm dec}}$ |
| Support | Compact: $\ell + 2\tau_s$ | Infinite (exponential tails) |
| $\hat{\Gamma}(\omega)$ | Closed-form, real (Eq. `Gamma_hat`) | Closed-form, complex |
| $R_\Gamma(\tau)$ | Piecewise degree-5 polynomial | Piecewise: exponentials + linear on plateau |
| Smoothness of $R_\Gamma$ | Kinks at $\tau_s$, $\ell$, $\ell+\tau_s$ | Kink at $\tau_{\rm plat}$ |
| Smoothness class | $C^1$ envelope, $C^0$ at ramp boundaries | $C^0$ (kinks at $t=0$, $t=\tau_{\rm plat}$) |
| Asymmetric rise/decay | Not supported | Naturally supported |
| $R_\Gamma(0)$ | $\alpha_{\max}^4(\ell + 2\tau_s/5)$ | $\alpha_{\max}^4(\tau_{\rm plat} + \tau_{\rm em}/4 + \tau_{\rm dec}/4)$ |

The exponential envelope provides an analytically tractable alternative to the trapezoidal
model, with the additional flexibility of asymmetric emergence and decay timescales.
However, its $C^0$ nature means $R_\Gamma(\tau)$ inherits derivative discontinuities---the
sigmoid envelope (Eq. 11) is preferred when smoothness is required.